In [ ]:
import pandas as pd
import torch
import triton

In [ ]:
# nn.LSTM and Graph sometimes go oom -- drop all nans for this run to avoid misleading summary stats in describe
df1 = pd.read_parquet("../full_fp32.parquet").dropna().reset_index().query("level_0!='s5'")
df2 = pd.read_parquet("../full_bf16.parquet").dropna()
df3 = pd.read_parquet("../full_fp16.parquet").dropna().reset_index().query("level_0!='s5'")

display(df1.describe())
display(df2.describe())
display(df3.describe())


In [ ]:
# forward runtime visualisation
df_in = pd.read_parquet("../fwd_bf16.parquet").dropna()
dfr = df_in / df_in["lstm"].to_numpy().reshape(-1, 1)

df_fwd = (dfr
          ["fast"]
          .droplevel(-1)
          .unstack(-1)
)

df_fwd.columns = pd.MultiIndex.from_tuples(
    [(f"FWD_bf16: FastLSTM/nn.LSTM -- torch:{torch.__version__} -- triton:{triton.__version__} -- RTX 2000 Ada" ,f"hidden:{64 << int(c[1:])}") for c in df_fwd]
)

df_fwd.index = pd.MultiIndex.from_tuples(
    [(f"seq:{64<<int(s[1:])}", f"batch:{4 << int(b[1:])}") for s, b in df_fwd.index]
)

df_fwd = df_fwd.style.background_gradient(cmap="bwr", vmin=1-1, vmax=1+1).format("{:.2f}")

df_fwd

In [ ]:
## Manual tuning
from fastlstm.lstm import FastLSTM, PersistentLSTMfn
import torch
import torch.nn as nn 

from triton.testing import do_bench

hidden_size = 256

m = nn.LSTM(input_size=hidden_size,
           hidden_size=hidden_size,
           device="cuda",
        #    version="persistent",
           )


x = torch.randn((128, 64, hidden_size), device="cuda")

res = {}

base = {"BLOCK_SIZE_H": 32, "BLOCK_SIZE_B": 32, "BLOCK_SIZE_K": 32, "num_warps": 2, "num_stages": 1}

configs = [
    ("vanilla", base),
    ("w1", base | {"num_warps": 1} ),
    ("w4", base | {"num_warps": 4} ),
    ("s2", base | {"num_warps": 1, "num_stages": 2} ),
    ("bigK", base | {"num_warps": 1, "BLOCK_SIZE_K": 16} ),
]

ref = 4.5522597537321206

for n, c in configs:
    PersistentLSTMfn.BWD_TRITON_CONFIG = c
    res[n] = do_bench(lambda : m(x)[0].sum().backward())/ref
    PersistentLSTMfn.BWD_TRITON_CONFIG = None

# 1.197

In [ ]:
from torch.amp import autocast
from fastlstm.lstm import FastLSTM, PersistentLSTMfn
import torch
import torch.nn as nn 

from triton.testing import do_bench

hidden_size = 256
x = torch.randn((1, 8, hidden_size), device="cuda")


if False:
    m = nn.LSTM(input_size=hidden_size,
            hidden_size=hidden_size,
            device="cuda",
            #    version="persistent",
            )
else:
    m = FastLSTM(input_size=hidden_size,
                 hidden_size=hidden_size,
                 device="cuda",
                 version="persistent",
                 )

def fn(dtype):
    dd = {
        "fp32": None,
        "fp16": torch.float16,
        "bf16": torch.bfloat16
    }
    with autocast("cuda", dd[dtype], enabled=dtype!="fp32"):
        m(x)[0].sum().backward()
        
y = x + 12
with autocast("cuda", torch.float16):
    print(y.dtype)
    t = m(y)[0]
    print(t.dtype)
    t.sum().backward()
    for n, p in m.named_parameters():
        print(n, p.dtype, p.grad.dtype)
    print(y.dtype)
    

In [ ]:
import pandas as pd

ax = (pd.DataFrame(res, columns=["batch_size", "nn.LSTM", "persistent", "graph", "cuda_fused", "cuda"])
 .set_index("batch_size")
 [["nn.LSTM", "graph", "persistent", "Flash:alternating", "Flash:fused"]]
 .plot(kind="bar", ylabel="mean runtime [ms]", title="fp16 FWD pass on H100 -- seq=1024, hidden=768", grid=True)
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.1f")

ax


In [ ]:
# visualise overfit losses
import pandas as pd

df = pd.read_parquet("../bf16_losses.parquet")
df.plot(logy=True)


In [ ]:
from collections import defaultdict
import json
import re

import numpy as np

with open("../graph_tune.json") as fh:
    data = json.load(fh)

res = defaultdict(list)

for setting, configs in data.items():
    for config, perf in configs:
        matches = re.findall("BLOCK_SIZE_H: ([0-9]+), BLOCK_SIZE_B: ([0-9]+), BLOCK_SIZE_K: ([0-9]+), GROUP_SIZE_B: ([0-9]+), num_warps: ([0-9]+), num_ctas: 1, num_stages: ([0-9]+), maxnreg: None", config)
        assert len(matches) == 1
        h, b, k, g, w, s = (int(e) for e in matches[0])
        
        res[(setting, h, b ,k, g, w, s)] += [perf[1]]

for k, v in res.items():
    res[k] = np.array(v).mean()


raw = (pd.DataFrame(res, index=["val"])
 .stack(0)
 .T
 .droplevel(0, axis=1)
)

best = pd.concat([raw.idxmin(), 1_000*raw.min()], axis=1, keys=["config", "time"])

